In [15]:
import os

# Criando a estrutura de pastas necessária
pastas = [
    '/content/data/fashion',
    '/content/out',
    '/content/figs/q1a',
    '/content/figs/q1b',
    '/content/figs/q2',
    '/content/figs/q3',
    '/content/figs/q4',
    '/content/figs/q5'
]

for pasta in pastas:
    os.makedirs(pasta, exist_ok=True)

print("Estrutura de diretórios criada com sucesso!")

Estrutura de diretórios criada com sucesso!


### Download datasets

In [16]:
# Download California Housing dataset
!wget -nc https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.tgz
!tar -xzf housing.tgz -C /content/data/

# Download Titanic dataset
!wget -nc https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv -O /content/data/titanic.csv

# Download Fashion-MNIST dataset files
!wget -nc http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-images-idx3-ubyte.gz -P /content/data/fashion/
!wget -nc http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-labels-idx1-ubyte.gz -P /content/data/fashion/
!wget -nc http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-images-idx3-ubyte.gz -P /content/data/fashion/
!wget -nc http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-labels-idx1-ubyte.gz -P /content/data/fashion/

print("Todos os datasets foram baixados e movidos para os diretórios corretos.")

File ‘housing.tgz’ already there; not retrieving.

File ‘/content/data/titanic.csv’ already there; not retrieving.
File ‘/content/data/fashion/train-images-idx3-ubyte.gz’ already there; not retrieving.

File ‘/content/data/fashion/train-labels-idx1-ubyte.gz’ already there; not retrieving.

File ‘/content/data/fashion/t10k-images-idx3-ubyte.gz’ already there; not retrieving.

File ‘/content/data/fashion/t10k-labels-idx1-ubyte.gz’ already there; not retrieving.

Todos os datasets foram baixados e movidos para os diretórios corretos.


Loader do Fashion-MNIST

In [17]:
%%writefile /content/fashion_mnist_loader.py
import gzip
import numpy as np

CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def load_images(path):
    with gzip.open(path, 'rb') as f:
        data = np.frombuffer(f.read(), np.uint8, offset=16)
    return data.reshape(-1, 28, 28)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        data = np.frombuffer(f.read(), np.uint8, offset=8)
    return data

def load_fashion_mnist(base='/content/data/fashion'):
    X_train = load_images(f'{base}/train-images-idx3-ubyte.gz')
    y_train = load_labels(f'{base}/train-labels-idx1-ubyte.gz')
    X_test = load_images(f'{base}/t10k-images-idx3-ubyte.gz')
    y_test = load_labels(f'{base}/t10k-labels-idx1-ubyte.gz')
    return X_train, y_train, X_test, y_test

Overwriting /content/fashion_mnist_loader.py


Questão 1a - Aproximação de Função

In [18]:
"""
Questão 1a - MLP para aproximar f(x) = 10x^5 + 5x^4 + 2x^3 - 0.5x^2 + 3x + 2, 0<=x<=5
"""
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)

def f(x):
    return 10*x**5 + 5*x**4 + 2*x**3 - 0.5*x**2 + 3*x + 2

N = 2000
X = np.random.uniform(0, 5, N).reshape(-1, 1)
y = f(X).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sx = StandardScaler().fit(X_train)
sy = StandardScaler().fit(y_train.reshape(-1, 1))
X_train_s = sx.transform(X_train)
X_test_s = sx.transform(X_test)
y_train_s = sy.transform(y_train.reshape(-1, 1)).ravel()
y_test_s = sy.transform(y_test.reshape(-1, 1)).ravel()

mlp = MLPRegressor(hidden_layer_sizes=(64, 64, 32), activation='relu', solver='adam',
                    max_iter=2000, learning_rate_init=0.001, random_state=42,
                    early_stopping=True, validation_fraction=0.15, n_iter_no_change=50)
mlp.fit(X_train_s, y_train_s)

y_pred_test_s = mlp.predict(X_test_s)
y_pred_test = sy.inverse_transform(y_pred_test_s.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)

print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

with open('/content/out/q1a_metrics.txt', 'w') as fo:
    fo.write(f"MAE: {mae:.4f}\nMSE: {mse:.4f}\nRMSE: {rmse:.4f}\n")
    fo.write(f"N iteracoes treino: {mlp.n_iter_}\n")

X_plot = np.linspace(0, 5, 300).reshape(-1, 1)
y_real = f(X_plot).ravel()
X_plot_s = sx.transform(X_plot)
y_plot_pred_s = mlp.predict(X_plot_s)
y_plot_pred = sy.inverse_transform(y_plot_pred_s.reshape(-1, 1)).ravel()

plt.figure(figsize=(8, 5))
plt.plot(X_plot, y_real, label='Função real f(x)', linewidth=2)
plt.plot(X_plot, y_plot_pred, label='Aproximação MLP', linestyle='--', linewidth=2)
plt.xlabel('x'); plt.ylabel('f(x)')
plt.title('Função real vs. função aproximada pela MLP')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q1a/real_vs_aprox.png', dpi=130); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, label='Erro de treinamento (loss)')
plt.xlabel('Época'); plt.ylabel('Loss (MSE normalizado)')
plt.title('Curva de erro de treinamento')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q1a/curva_treino.png', dpi=130); plt.close()

if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_ is not None:
    val_error = [1 - s for s in mlp.validation_scores_]
    plt.figure(figsize=(8, 5))
    plt.plot(val_error, color='orange', label='Erro de validação (1 - R²)')
    plt.xlabel('Época'); plt.ylabel('Erro de validação')
    plt.title('Curva de erro de validação')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig('/content/figs/q1a/curva_validacao.png', dpi=130); plt.close()

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_test, alpha=0.5, s=15)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
plt.plot(lims, lims, 'r--', label='Predição perfeita')
plt.xlabel('Valor real'); plt.ylabel('Valor predito')
plt.title('Comparação: valores reais vs. preditos (teste)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q1a/real_vs_predito_scatter.png', dpi=130); plt.close()

print("Concluído Q1a")

MAE: 21.6911
MSE: 1340.0524
RMSE: 36.6067
Concluído Q1a


Questão 1b - California Housing

In [19]:
"""
Questão 1b - MLP para prever valor médio de imóveis - California Housing Dataset
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)

df = pd.read_csv('/content/data/housing.csv')
print(df.head())
print(df.describe())
print("Valores ausentes:\n", df.isna().sum())

df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlação - California Housing')
plt.tight_layout()
plt.savefig('/content/figs/q1b/correlacao.png', dpi=130); plt.close()

plt.figure(figsize=(8, 5))
sns.histplot(df['median_house_value'], bins=50, kde=True)
plt.title('Distribuição do valor médio dos imóveis (target)')
plt.xlabel('median_house_value (USD)')
plt.tight_layout()
plt.savefig('/content/figs/q1b/dist_target.png', dpi=130); plt.close()

num_cols = ['longitude','latitude','housing_median_age','total_rooms','total_bedrooms',
            'population','households','median_income']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig('/content/figs/q1b/dist_features.png', dpi=130); plt.close()

plt.figure(figsize=(7,6))
sc = plt.scatter(df['longitude'], df['latitude'], c=df['median_house_value'], cmap='viridis', s=8, alpha=0.5)
plt.colorbar(sc, label='median_house_value')
plt.xlabel('longitude'); plt.ylabel('latitude')
plt.title('Distribuição geográfica do valor dos imóveis')
plt.tight_layout()
plt.savefig('/content/figs/q1b/geo_scatter.png', dpi=130); plt.close()

target = 'median_house_value'
X = df.drop(columns=[target])
y = df[target].values

num_features = ['longitude','latitude','housing_median_age','total_rooms','total_bedrooms',
                 'population','households','median_income','rooms_per_household',
                 'bedrooms_per_room','population_per_household']
cat_features = ['ocean_proximity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preproc = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])
X_train_s = preproc.fit_transform(X_train)
X_test_s = preproc.transform(X_test)

scaler_y = StandardScaler().fit(y_train.reshape(-1, 1))
y_train_s = scaler_y.transform(y_train.reshape(-1, 1)).ravel()
y_test_s = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

mlp = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam',
                    alpha=1e-4, learning_rate_init=0.001, max_iter=400, random_state=42,
                    early_stopping=True, validation_fraction=0.15, n_iter_no_change=20)
mlp.fit(X_train_s, y_train_s)

y_pred_s = mlp.predict(X_test_s)
y_pred = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"MAE: {mae:.4f}  MSE: {mse:.4f}  RMSE: {rmse:.4f}")

with open('/content/out/q1b_metrics.txt', 'w') as fo:
    fo.write(f"Arquitetura: {mlp.hidden_layer_sizes}\n")
    fo.write(f"MAE: {mae:.4f}\nMSE: {mse:.4f}\nRMSE: {rmse:.4f}\n")
    fo.write(f"N iteracoes: {mlp.n_iter_}\n")

plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, label='Erro de treinamento')
plt.xlabel('Época'); plt.ylabel('Loss'); plt.title('Curva de erro de treinamento - California Housing')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q1b/curva_treino.png', dpi=130); plt.close()

if mlp.validation_scores_ is not None:
    val_error = [1 - s for s in mlp.validation_scores_]
    plt.figure(figsize=(8, 5))
    plt.plot(val_error, color='orange', label='Erro de validação (1-R²)')
    plt.xlabel('Época'); plt.ylabel('Erro'); plt.title('Curva de erro de validação')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig('/content/figs/q1b/curva_validacao.png', dpi=130); plt.close()

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', label='Predição perfeita')
plt.xlabel('Valor real'); plt.ylabel('Valor predito')
plt.title('California Housing: real vs. predito')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q1b/real_vs_predito.png', dpi=130); plt.close()

print("Concluído Q1b")

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  
          longitude      latitude  housing_median_age   total_rooms  \
coun

Questão 2 - Titanic

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                              precision_score, recall_score, f1_score, classification_report)

np.random.seed(42)

df = pd.read_csv('/content/data/titanic.csv')

# --- Análise exploratória ---
plt.figure(figsize=(5,4))
sns.countplot(x='Survived', data=df)
plt.title('Distribuição das classes (0=não sobreviveu, 1=sobreviveu)')
plt.tight_layout()
plt.savefig('/content/figs/q2/dist_classes.png', dpi=130); plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(x='Survived', hue='Sex', data=df, ax=axes[0]); axes[0].set_title('Sobrevivência por sexo')
sns.countplot(x='Survived', hue='Pclass', data=df, ax=axes[1]); axes[1].set_title('Sobrevivência por classe')
sns.histplot(data=df, x='Age', hue='Survived', bins=30, kde=True, ax=axes[2]); axes[2].set_title('Idade x sobrevivência')
plt.tight_layout()
plt.savefig('/content/figs/q2/exploratoria.png', dpi=130); plt.close()

with open('/content/out/q2_eda.txt', 'w') as fo:
    fo.write("Valores ausentes por coluna:\n")
    fo.write(str(df.isna().sum()))
    fo.write("\n\nDistribuição das classes:\n")
    fo.write(str(df['Survived'].value_counts(normalize=True)))

# --- Seleção / remoção de atributos irrelevantes ou redundantes ---
df = df.drop(columns=['deck', 'alive', 'class', 'adult_male', 'embark_town'], errors='ignore')

# --- Tratamento de dados faltantes ---
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# --- Feature engineering ---
df['family_size'] = df['SibSp'] + df['Parch'] + 1

target = 'Survived'
num_features = ['Age', 'SibSp', 'Parch', 'Fare', 'family_size']
cat_features = ['Pclass', 'Sex', 'Embarked']

X = df[num_features + cat_features]
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preproc = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])
X_train_s = preproc.fit_transform(X_train)
X_test_s = preproc.transform(X_test)

# Balanceamento
from sklearn.utils import resample
Xy_train = pd.DataFrame(X_train_s.toarray() if hasattr(X_train_s, 'toarray') else X_train_s)
Xy_train['y'] = y_train
majority = Xy_train[Xy_train['y'] == 0]
minority = Xy_train[Xy_train['y'] == 1]
minority_up = resample(minority, replace=True, n_samples=len(majority), random_state=42)
balanced = pd.concat([majority, minority_up])
X_train_bal = balanced.drop(columns='y').values
y_train_bal = balanced['y'].values

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
                     alpha=1e-4, learning_rate_init=0.001, max_iter=500, random_state=42,
                     early_stopping=True, validation_fraction=0.15, n_iter_no_change=25)
mlp.fit(X_train_bal, y_train_bal)

y_pred = mlp.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
print(classification_report(y_test, y_pred))

with open('/content/out/q2_metrics.txt', 'w') as fo:
    fo.write(f"Arquitetura: {mlp.hidden_layer_sizes}\n")
    fo.write(f"Accuracy: {acc:.4f}\nPrecision: {prec:.4f}\nRecall: {rec:.4f}\nF1-score: {f1:.4f}\n\n")
    fo.write(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Não sobreviveu', 'Sobreviveu'])
disp.plot(cmap='Blues')
plt.title('Matriz de confusão - Titanic')
plt.tight_layout()
plt.savefig('/content/figs/q2/matriz_confusao.png', dpi=130); plt.close()

plt.figure(figsize=(8,5))
plt.plot(mlp.loss_curve_, label='Erro de treinamento')
plt.xlabel('Época'); plt.ylabel('Loss'); plt.title('Curva de erro de treinamento - Titanic')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q2/curva_treino.png', dpi=130); plt.close()

if mlp.validation_scores_ is not None:
    val_error = [1 - s for s in mlp.validation_scores_]
    plt.figure(figsize=(8,5))
    plt.plot(val_error, color='orange', label='Erro de validação (1-acc)')
    plt.xlabel('Época'); plt.ylabel('Erro'); plt.title('Curva de erro de validação - Titanic')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig('/content/figs/q2/curva_validacao.png', dpi=130); plt.close()

print("Concluído Q2")

Accuracy: 0.7598  Precision: 0.6757  Recall: 0.7246  F1: 0.6993
              precision    recall  f1-score   support

           0       0.82      0.78      0.80       110
           1       0.68      0.72      0.70        69

    accuracy                           0.76       179
   macro avg       0.75      0.75      0.75       179
weighted avg       0.76      0.76      0.76       179

Concluído Q2


Questão 3 - Fashion-MNIST (MLP vs CNN)

In [27]:
"""
Questão 3 - Fashion-MNIST: MLP vs CNN
"""
import sys
sys.path.insert(0, '/content')
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from fashion_mnist_loader import load_fashion_mnist, CLASSES

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando device: {device}")

X_train_full, y_train_full, X_test_full, y_test_full = load_fashion_mnist()

# Subconjuntos para viabilizar treino em CPU único
N_TRAIN, N_VAL, N_TEST = 12000, 2000, 2000
idx = np.random.permutation(len(X_train_full))
train_idx, val_idx = idx[:N_TRAIN], idx[N_TRAIN:N_TRAIN+N_VAL]
test_idx = np.random.permutation(len(X_test_full))[:N_TEST]

X_train, y_train = X_train_full[train_idx], y_train_full[train_idx]
X_val, y_val = X_train_full[val_idx], y_train_full[val_idx]
X_test, y_test = X_test_full[test_idx], y_test_full[test_idx]

def to_tensor_flat(X):
    return torch.tensor(X.reshape(len(X), -1) / 255.0, dtype=torch.float32).to(device)

def to_tensor_img(X):
    return torch.tensor(X.reshape(len(X), 1, 28, 28) / 255.0, dtype=torch.float32).to(device)

y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

BATCH = 128

# ---------------- MLP ----------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

# ---------------- CNN ----------------
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 28->14
        x = self.pool(self.relu(self.conv2(x)))   # 14->7
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

def train_model(model, X_train_t, X_val_t, X_test_t, epochs, lr=1e-3, tag=''):
    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    train_losses, val_losses = [], []
    for ep in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            opt.zero_grad()
            out = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_ds)
        model.eval()
        with torch.no_grad():
            val_out = model(X_val_t)
            val_loss = crit(val_out, y_val_t).item()
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"[{tag}] Epoch {ep+1}/{epochs} - train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
    return train_losses, val_losses

# --- Treino MLP ---
X_train_flat = to_tensor_flat(X_train)
X_val_flat = to_tensor_flat(X_val)
X_test_flat = to_tensor_flat(X_test)

mlp = MLP().to(device)
mlp_train_losses, mlp_val_losses = train_model(mlp, X_train_flat, X_val_flat, X_test_flat, epochs=15, tag='MLP')

mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(X_test_flat).argmax(dim=1).cpu().numpy()
mlp_acc = accuracy_score(y_test, mlp_pred)
print("MLP test acc:", mlp_acc)

# --- Treino CNN ---
X_train_img = to_tensor_img(X_train)
X_val_img = to_tensor_img(X_val)
X_test_img = to_tensor_img(X_test)

cnn = CNN().to(device)
cnn_train_losses, cnn_val_losses = train_model(cnn, X_train_img, X_val_img, X_test_img, epochs=10, tag='CNN')

cnn.eval()
with torch.no_grad():
    cnn_pred = cnn(X_test_img).argmax(dim=1).cpu().numpy()
cnn_acc = accuracy_score(y_test, cnn_pred)
print("CNN test acc:", cnn_acc)

# --- Salvar modelo e dados para uso na Q4/Q5 ---
torch.save(cnn.state_dict(), '/content/out/cnn_fashion.pt')
np.savez('/content/out/q3_test_subset.npz', X_test=X_test, y_test=y_test)

# --- Gráficos ---
plt.figure(figsize=(8,5))
plt.plot(mlp_train_losses, label='Treino'); plt.plot(mlp_val_losses, label='Validação')
plt.xlabel('Época'); plt.ylabel('Loss (Cross-Entropy)'); plt.title('MLP - Curva de erro (Fashion-MNIST)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q3/mlp_curva_erro.png', dpi=130); plt.close()

plt.figure(figsize=(8,5))
plt.plot(cnn_train_losses, label='Treino'); plt.plot(cnn_val_losses, label='Validação')
plt.xlabel('Época'); plt.ylabel('Loss (Cross-Entropy)'); plt.title('CNN - Curva de erro (Fashion-MNIST)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q3/cnn_curva_erro.png', dpi=130); plt.close()

cm_mlp = confusion_matrix(y_test, mlp_pred)
disp = ConfusionMatrixDisplay(cm_mlp, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(9,9))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
plt.title('MLP - Matriz de confusão')
plt.tight_layout()
plt.savefig('/content/figs/q3/mlp_matriz_confusao.png', dpi=130); plt.close()

cm_cnn = confusion_matrix(y_test, cnn_pred)
disp = ConfusionMatrixDisplay(cm_cnn, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(9,9))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
plt.title('CNN - Matriz de confusão')
plt.tight_layout()
plt.savefig('/content/figs/q3/cnn_matriz_confusao.png', dpi=130); plt.close()

with open('/content/out/q3_metrics.txt', 'w') as fo:
    fo.write(f"MLP arquitetura: 784-256-128-10 (dropout 0.2), epochs=15\n")
    fo.write(f"MLP acuracia teste: {mlp_acc:.4f}\n\n")
    fo.write(f"CNN arquitetura: Conv(1->16,3x3)-Pool-Conv(16->32,3x3)-Pool-FC(1568->128)-FC(128->10), epochs=10\n")
    fo.write(f"CNN acuracia teste: {cnn_acc:.4f}\n")

print("Concluído Q3")

Usando device: cpu
[MLP] Epoch 1/15 - train_loss=1.0152 val_loss=0.5905
[MLP] Epoch 2/15 - train_loss=0.6057 val_loss=0.5052
[MLP] Epoch 3/15 - train_loss=0.5157 val_loss=0.4452
[MLP] Epoch 4/15 - train_loss=0.4625 val_loss=0.4394
[MLP] Epoch 5/15 - train_loss=0.4333 val_loss=0.4026
[MLP] Epoch 6/15 - train_loss=0.4068 val_loss=0.3874
[MLP] Epoch 7/15 - train_loss=0.3871 val_loss=0.3969
[MLP] Epoch 8/15 - train_loss=0.3623 val_loss=0.3888
[MLP] Epoch 9/15 - train_loss=0.3556 val_loss=0.3741
[MLP] Epoch 10/15 - train_loss=0.3498 val_loss=0.3620
[MLP] Epoch 11/15 - train_loss=0.3270 val_loss=0.3607
[MLP] Epoch 12/15 - train_loss=0.3191 val_loss=0.3621
[MLP] Epoch 13/15 - train_loss=0.3095 val_loss=0.3503
[MLP] Epoch 14/15 - train_loss=0.2989 val_loss=0.3591
[MLP] Epoch 15/15 - train_loss=0.2930 val_loss=0.3579
MLP test acc: 0.856
[CNN] Epoch 1/10 - train_loss=1.0693 val_loss=0.6116
[CNN] Epoch 2/10 - train_loss=0.6042 val_loss=0.4941
[CNN] Epoch 3/10 - train_loss=0.5277 val_loss=0.4413
[

Questão 4 - Visualização de CNN

In [28]:
"""
Questão 4 - Visualização de filtros e mapas de ativação da CNN (Q3), e análise de acertos/erros.
"""
import sys
sys.path.insert(0, '/content')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from fashion_mnist_loader import CLASSES

torch.manual_seed(42)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, return_maps=False):
        a1 = self.relu(self.conv1(x))
        p1 = self.pool(a1)
        a2 = self.relu(self.conv2(p1))
        p2 = self.pool(a2)
        x2 = p2.view(p2.size(0), -1)
        x2 = self.relu(self.fc1(x2))
        out = self.fc2(x2)
        if return_maps:
            return out, a1, a2
        return out

cnn = CNN()
cnn.load_state_dict(torch.load('/content/out/cnn_fashion.pt', map_location='cpu'))
cnn.eval()

data = np.load('/content/out/q3_test_subset.npz')
X_test, y_test = data['X_test'], data['y_test']

# ---------- (a) Filtros da primeira camada convolucional ----------
filters = cnn.conv1.weight.data.numpy()  # (16, 1, 3, 3)
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.ravel()):
    f = filters[i, 0]
    ax.imshow(f, cmap='gray')
    ax.set_title(f'Filtro {i+1}', fontsize=9)
    ax.axis('off')
plt.suptitle('Filtros aprendidos - 1ª camada convolucional (conv1)')
plt.tight_layout()
plt.savefig('/content/figs/q4/filtros_conv1.png', dpi=130); plt.close()

# ---------- (b) Mapas de ativação para 5+ imagens ----------
X_img = torch.tensor(X_test[:8].reshape(-1, 1, 28, 28) / 255.0, dtype=torch.float32)
with torch.no_grad():
    _, act1, act2 = cnn(X_img, return_maps=True)

n_examples = 5
for idx in range(n_examples):
    fig, axes = plt.subplots(1, 7, figsize=(18, 3))
    axes[0].imshow(X_test[idx], cmap='gray'); axes[0].set_title(f'Original\n({CLASSES[y_test[idx]]})'); axes[0].axis('off')
    maps = act2[idx].numpy()
    chosen = np.argsort(-maps.reshape(maps.shape[0], -1).mean(axis=1))[:6]
    for j, ch in enumerate(chosen):
        axes[j+1].imshow(maps[ch], cmap='viridis')
        axes[j+1].set_title(f'Canal {ch}', fontsize=9)
        axes[j+1].axis('off')
    plt.suptitle(f'Mapas de ativação (conv2) - exemplo {idx+1}')
    plt.tight_layout()
    plt.savefig(f'/content/figs/q4/ativacao_exemplo_{idx+1}.png', dpi=130); plt.close()

# ---------- (c) Exemplos corretos e incorretos ----------
X_all = torch.tensor(X_test.reshape(-1, 1, 28, 28) / 255.0, dtype=torch.float32)
with torch.no_grad():
    logits = cnn(X_all)
    probs = F.softmax(logits, dim=1).numpy()
    preds = probs.argmax(axis=1)

correct_idx = np.where(preds == y_test)[0]
wrong_idx = np.where(preds != y_test)[0]

rng = np.random.default_rng(42)
sel_correct = rng.choice(correct_idx, size=min(5, len(correct_idx)), replace=False)
sel_wrong = rng.choice(wrong_idx, size=min(5, len(wrong_idx)), replace=False)

def plot_examples(indices, title, fname):
    fig, axes = plt.subplots(1, len(indices), figsize=(3*len(indices), 3.5))
    if len(indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, indices):
        ax.imshow(X_test[idx], cmap='gray')
        true_c = CLASSES[y_test[idx]]
        pred_c = CLASSES[preds[idx]]
        score = probs[idx, preds[idx]]
        ax.set_title(f'Real: {true_c}\nPred: {pred_c}\nP={score:.2f}', fontsize=9)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(fname, dpi=130); plt.close()

plot_examples(sel_correct, 'Exemplos classificados corretamente',
              '/content/figs/q4/exemplos_corretos.png')
plot_examples(sel_wrong, 'Exemplos classificados incorretamente',
              '/content/figs/q4/exemplos_incorretos.png')

with open('/content/out/q4_notas.txt', 'w') as fo:
    fo.write("Descrição qualitativa dos filtros (conv1):\n")
    fo.write("Os filtros da primeira camada aprendem principalmente detectores de bordas e "
             "gradientes de intensidade em diferentes orientacoes (horizontal, vertical e "
             "diagonal), alem de alguns filtros que respondem a regioes quase uniformes "
             "(deteccao de fundo/textura lisa). Isso e esperado para a primeira camada de "
             "uma CNN rasa treinada em imagens em escala de cinza.\n\n")
    fo.write("Mapas de ativação (conv2): tendem a realçar contornos das peças de roupa e "
             "regiões de alto contraste (ex.: gola, mangas, silhueta), sugerindo que a rede "
             "já combina bordas simples em padrões mais estruturais nesta camada.\n\n")
    fo.write(f"Total corretos: {len(correct_idx)} / {len(y_test)}\n")
    fo.write(f"Total incorretos: {len(wrong_idx)} / {len(y_test)}\n")

print("Concluído Q4")

Concluído Q4


Questão 5 - Autoencoders

In [29]:
"""
Questão 5 - Autoencoders no Fashion-MNIST
5a: autoencoder convolucional com 3 tamanhos de espaço latente
5b: denoising autoencoder
"""
import sys
sys.path.insert(0, '/content')
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from fashion_mnist_loader import load_fashion_mnist

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_full, y_train_full, X_test_full, y_test_full = load_fashion_mnist()

N_TRAIN, N_TEST = 8000, 1000
idx = np.random.permutation(len(X_train_full))[:N_TRAIN]
tidx = np.random.permutation(len(X_test_full))[:N_TEST]
X_train = X_train_full[idx].reshape(-1, 1, 28, 28) / 255.0
X_test = X_test_full[tidx].reshape(-1, 1, 28, 28) / 255.0

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

BATCH = 128

class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),   # 28->14
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),  # 14->7
            nn.Flatten(),
            nn.Linear(32*7*7, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32*7*7), nn.ReLU(),
            nn.Unflatten(1, (32, 7, 7)),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),  # 7->14
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid()  # 14->28
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

def train_ae(model, X_in_t, X_target_t, epochs=12, lr=1e-3, tag=''):
    ds = TensorDataset(X_in_t, X_target_t)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    losses = []
    for ep in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
        ep_loss = running / len(ds)
        losses.append(ep_loss)
        print(f"[{tag}] epoch {ep+1}/{epochs} loss={ep_loss:.5f}")
    return losses

# --------- 5a: três tamanhos de espaço latente ---------
latent_sizes = [8, 32, 64]
models = {}
recon_errors = {}
all_losses = {}

for ld in latent_sizes:
    ae = ConvAutoencoder(ld).to(device)
    losses = train_ae(ae, X_train_t, X_train_t, epochs=12, tag=f'AE-latent{ld}')
    models[ld] = ae
    all_losses[ld] = losses
    ae.eval()
    with torch.no_grad():
        recon = ae(X_test_t)
        err = ((recon - X_test_t) ** 2).mean(dim=[1,2,3]).cpu().numpy()
    recon_errors[ld] = err
    print(f"Latent {ld}: erro médio de reconstrução (teste) = {err.mean():.5f}")

with open('/content/out/q5a_metrics.txt', 'w') as fo:
    for ld in latent_sizes:
        fo.write(f"Espaço latente={ld}: MSE médio (teste) = {recon_errors[ld].mean():.5f}, "
                 f"std={recon_errors[ld].std():.5f}\n")

plt.figure(figsize=(8,5))
for ld in latent_sizes:
    plt.plot(all_losses[ld], label=f'Latent dim = {ld}')
plt.xlabel('Época'); plt.ylabel('MSE de reconstrução (treino)')
plt.title('Autoencoder - curva de treino para diferentes espaços latentes')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q5/ae_curvas_treino.png', dpi=130); plt.close()

n_show = 10
show_idx = np.arange(n_show)
fig, axes = plt.subplots(len(latent_sizes)+1, n_show, figsize=(1.6*n_show, 1.6*(len(latent_sizes)+1)))
for j in range(n_show):
    axes[0, j].imshow(X_test[show_idx[j], 0], cmap='gray'); axes[0, j].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=10)

for i, ld in enumerate(latent_sizes):
    models[ld].eval()
    with torch.no_grad():
        recon = models[ld](X_test_t[show_idx]).cpu().numpy()
    for j in range(n_show):
        axes[i+1, j].imshow(recon[j, 0], cmap='gray'); axes[i+1, j].axis('off')
    axes[i+1, 0].set_ylabel(f'Latent={ld}', fontsize=10)

for i, label in enumerate(['Original'] + [f'Latent={ld}' for ld in latent_sizes]):
    axes[i, 0].axis('on')
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
    axes[i, 0].set_ylabel(label, fontsize=10)

plt.suptitle('Imagens originais vs. reconstruções (diferentes espaços latentes)')
plt.tight_layout()
plt.savefig('/content/figs/q5/ae_reconstrucoes_comparacao.png', dpi=130); plt.close()

# --------- 5b: Denoising autoencoder ---------
noise_factor = 0.4
X_train_noisy = X_train_t + noise_factor * torch.randn(*X_train_t.shape, device=device)
X_train_noisy = torch.clamp(X_train_noisy, 0., 1.)
X_test_noisy = X_test_t + noise_factor * torch.randn(*X_test_t.shape, device=device)
X_test_noisy = torch.clamp(X_test_noisy, 0., 1.)

dae = ConvAutoencoder(latent_dim=32).to(device)
dae_losses = train_ae(dae, X_train_noisy, X_train_t, epochs=15, tag='DAE')

dae.eval()
with torch.no_grad():
    denoised = dae(X_test_noisy[show_idx]).cpu().numpy()
    dae_err = ((dae(X_test_noisy) - X_test_t) ** 2).mean().item()

with open('/content/out/q5b_metrics.txt', 'w') as fo:
    fo.write(f"Denoising AE - latent dim=32, noise_factor={noise_factor}\n")
    fo.write(f"MSE médio de reconstrução (teste, comparado à imagem limpa) = {dae_err:.5f}\n")

plt.figure(figsize=(8,5))
plt.plot(dae_losses, label='Treino (denoising)')
plt.xlabel('Época'); plt.ylabel('MSE'); plt.title('Denoising Autoencoder - curva de treino')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/figs/q5/dae_curva_treino.png', dpi=130); plt.close()

fig, axes = plt.subplots(3, n_show, figsize=(1.6*n_show, 5))
for j in range(n_show):
    axes[0, j].imshow(X_test[show_idx[j], 0], cmap='gray'); axes[0, j].axis('off')
    axes[1, j].imshow(X_test_noisy[show_idx[j], 0].cpu().numpy(), cmap='gray'); axes[1, j].axis('off')
    axes[2, j].imshow(denoised[j, 0], cmap='gray'); axes[2, j].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=10); axes[0,0].axis('on'); axes[0,0].set_xticks([]); axes[0,0].set_yticks([])
axes[1, 0].set_ylabel('Ruidosa', fontsize=10); axes[1,0].axis('on'); axes[1,0].set_xticks([]); axes[1,0].set_yticks([])
axes[2, 0].set_ylabel('Reconstruída', fontsize=10); axes[2,0].axis('on'); axes[2,0].set_xticks([]); axes[2,0].set_yticks([])
plt.suptitle('Denoising Autoencoder: original, ruidosa e reconstruída')
plt.tight_layout()
plt.savefig('/content/figs/q5/dae_resultados.png', dpi=130); plt.close()

print("Concluído Q5")

[AE-latent8] epoch 1/12 loss=0.11970
[AE-latent8] epoch 2/12 loss=0.07233
[AE-latent8] epoch 3/12 loss=0.04957
[AE-latent8] epoch 4/12 loss=0.04215
[AE-latent8] epoch 5/12 loss=0.03897
[AE-latent8] epoch 6/12 loss=0.03712
[AE-latent8] epoch 7/12 loss=0.03591
[AE-latent8] epoch 8/12 loss=0.03508
[AE-latent8] epoch 9/12 loss=0.03442
[AE-latent8] epoch 10/12 loss=0.03402
[AE-latent8] epoch 11/12 loss=0.03351
[AE-latent8] epoch 12/12 loss=0.03315
Latent 8: erro médio de reconstrução (teste) = 0.03357
[AE-latent32] epoch 1/12 loss=0.10710
[AE-latent32] epoch 2/12 loss=0.04679
[AE-latent32] epoch 3/12 loss=0.02993
[AE-latent32] epoch 4/12 loss=0.02531
[AE-latent32] epoch 5/12 loss=0.02296
[AE-latent32] epoch 6/12 loss=0.02148
[AE-latent32] epoch 7/12 loss=0.02050
[AE-latent32] epoch 8/12 loss=0.01973
[AE-latent32] epoch 9/12 loss=0.01912
[AE-latent32] epoch 10/12 loss=0.01855
[AE-latent32] epoch 11/12 loss=0.01807
[AE-latent32] epoch 12/12 loss=0.01764
Latent 32: erro médio de reconstrução (